In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "transactions.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sergionefedov/fraud-detection-1m-transactions-7-fraud-types",
  file_path,
)

/tmp/ipykernel_1329/3901280428.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 122M/122M [00:03<00:00, 41.4MB/s]


## Importation des bibliothèques

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from google.colab import drive
import joblib
import os

#drive.mount('/content/drive')

In [4]:
base_path   = '/content/drive/MyDrive/fraud_detection'
data_path   = f'{base_path}/data'
models_path = f'{base_path}/models'

os.makedirs(data_path,   exist_ok=True)
os.makedirs(models_path, exist_ok=True)

## Données cibles

In [5]:
y_detection      = df['is_fraud']
y_classification = df['fraud_pattern']

## Suppression des colonnes

In [6]:
cols_a_supprimer = [
    'transaction_id',
    'account_id',
    'timestamp',
    'is_weekend',
    'amount',
    'is_fraud',
    'fraud_pattern',
]

df_clean = df.drop(columns=cols_a_supprimer)

In [8]:
for col in list(df_clean.columns):
  print(col)

hour_of_day
day_of_week
merchant_category
mcc_code
merchant_country
card_present
device_type
device_known
ip_risk_score
is_foreign_txn
time_since_last_s
velocity_1h
amount_vs_avg_ratio
account_age_days
has_2fa
credit_limit


## Encodage des variables catégorielles

In [ ]:
cols_categorielle = ['merchant_category', 'merchant_country', 'device_type']
cols_numerique    = [col for col in df_clean.columns if col not in cols_categorielle]

X_num = df_clean[cols_numerique].values
X_cat = df_clean[cols_categorielle].values

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X_cat)

In [ ]:
X = np.hstack([X_num, X_cat_encoded])

## Création des jeux de données (Celui contenant les cas normaux, celui contenant les fraudes, types de fraudes)

In [ ]:
X_normal = X[y_detection.values == 0]
X_fraude  = X[y_detection.values == 1]
y_fraude  = y_classification[y_detection.values == 1].values

## Standardisation des valeurs

In [ ]:
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_fraude_scaled = scaler.transform(X_fraude)
X_scaled    = scaler.transform(X)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Sauvegarde des jeux de données

In [ ]:
# Dataset standardisé complet
np.save(f'{data_path}/X_data.npy', X_scaled)
np.save(f'{data_path}/y_detection.npy', y_detection.values)

# Dataset normales (entraînement AE et IF)
np.save(f'{data_path}/X_normal.npy', X_normal_scaled)

# Dataset fraudes (classification)
np.save(f'{data_path}/X_fraude.npy', X_fraude_scaled)
np.save(f'{data_path}/y_fraude.npy', y_fraude)

# Modèles de preprocessing
joblib.dump(encoder, f'{model_path}/onehot_fraud_encoder.pkl')
joblib.dump(scaler, f'{model_path}/fraud_scaler.pkl')

['/content/drive/MyDrive/fraud_detection/models/fraud_scaler.pkl']